# Soran Population Forecasting — Clean Final Notebook

**Observed period:** 2015–2024  
**Forecast period:** 2025–2030

This notebook is a clean, reproducible version of the population-forecasting workflow. It reads the monthly climate workbook and the corrected population workbook, performs the annual climate aggregation and detrended climate–demography test, evaluates Polynomial, MLP, and ARIMAX models using expanding-window one-step-ahead validation, computes a small-sample Diebold–Mariano comparison, and produces final 2025–2030 forecasts with uncertainty intervals.

Only genuinely observed annual population values are used for validation. The monthly interpolation is used only inside the MLP training folds.

## 1. Setup

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path(r"D:\Kaka Paper")
CLIMATE_XLSX = BASE_DIR / "data" / "climate_monthly.xlsx"
POPULATION_XLSX = BASE_DIR / "data" / "population_annual.xlsx"
OUT_DIR = BASE_DIR / "files" / "outputs_clean"

(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print("Climate:", CLIMATE_XLSX)
print("Population:", POPULATION_XLSX)
print("Output:", OUT_DIR)

## 2. Load and clean monthly climate data

In [ ]:
def fix_december_labels(df, year_col="years", month_col="month"):
    years = pd.to_numeric(df[year_col], errors="coerce").astype("Int64").tolist()
    months = pd.to_numeric(df[month_col], errors="coerce").astype("Int64").tolist()
    fixed = years.copy()

    for i in range(len(df) - 1):
        if (months[i] == 12 and months[i + 1] == 1
                and years[i + 1] == years[i]):
            fixed[i] = years[i] - 1

    out = df.copy()
    out[year_col] = fixed
    out[month_col] = months
    return out.sort_values([year_col, month_col]).reset_index(drop=True)


def load_monthly_climate(path):
    raw = pd.read_excel(path, sheet_name="Data")
    raw = fix_december_labels(raw)

    required = ["years", "month", "c.avg", "humidity.avg", "precipition.depth"]
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise ValueError(f"Missing climate columns: {missing}")

    df = pd.DataFrame({
        "year": pd.to_numeric(raw["years"], errors="coerce"),
        "month": pd.to_numeric(raw["month"], errors="coerce"),
        "temp_avg_c": pd.to_numeric(raw["c.avg"], errors="coerce"),
        "humidity_avg_pct": pd.to_numeric(raw["humidity.avg"], errors="coerce"),
        "precip_mm": pd.to_numeric(raw["precipition.depth"], errors="coerce"),
    }).dropna(subset=["year", "month"])

    df["year"] = df["year"].astype(int)
    df["month"] = df["month"].astype(int)
    df = df.sort_values(["year", "month"]).reset_index(drop=True)

    dry = df["month"].isin([6, 7, 8, 9])
    df.loc[dry & df["precip_mm"].isna(), "precip_mm"] = 0.0
    df["precip_mm"] = df["precip_mm"].interpolate(limit_direction="both")
    df["temp_avg_c"] = df["temp_avg_c"].interpolate(limit_direction="both")

    return df

climate_m = load_monthly_climate(CLIMATE_XLSX)

print(f"Loaded {len(climate_m)} monthly climate rows.")
print(climate_m[["year","month","temp_avg_c","precip_mm"]].head())

In [ ]:
climate_annual = (
    climate_m.groupby("year")
    .agg(
        temp_avg_c=("temp_avg_c", "mean"),
        precip_total_mm=("precip_mm", "sum")
    )
    .reset_index()
)

climate_annual.to_csv(OUT_DIR / "tables" / "climate_annual.csv", index=False)
climate_annual

## 3. Load the corrected annual population input

In [ ]:
def load_population(path, years):
    raw = pd.read_excel(path, sheet_name="Data")
    required = ["years", "month", "population.soran"]
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise ValueError(f"Population workbook is missing: {missing}")

    raw = fix_december_labels(raw)

    raw["years"] = pd.to_numeric(raw["years"], errors="coerce")
    raw["month"] = pd.to_numeric(raw["month"], errors="coerce")
    raw["population.soran"] = pd.to_numeric(raw["population.soran"], errors="coerce")
    raw = raw.dropna(subset=["years", "month", "population.soran"])

    raw["years"] = raw["years"].astype(int)
    raw["month"] = raw["month"].astype(int)

    dec = raw.loc[raw["month"] == 12, ["years", "population.soran"]]
    annual = dec.groupby("years")["population.soran"].last().reindex(years)

    if annual.isna().any():
        raise ValueError(f"Missing annual population for: {annual[annual.isna()].index.tolist()}")

    return pd.Series(annual.values, index=years, name="population")

YEARS = list(range(2015, 2025))
population = load_population(POPULATION_XLSX, YEARS)

demo_annual = (
    pd.DataFrame({"year": YEARS, "population": population.values})
    .merge(climate_annual, on="year", how="left")
)

demo_annual.to_csv(OUT_DIR / "tables" / "population_annual_clean.csv", index=False)
demo_annual

## 4. Climate–demography association test

In [ ]:
from scipy.stats import spearmanr
from scipy.signal import detrend

def detrended_lagged_correlation(demo_df, clim_df, lags=(0, 1, 2)):
    d = (
        demo_df[["year", "population"]]
        .merge(clim_df, on="year")
        .sort_values("year")
        .reset_index(drop=True)
    )

    growth = d["population"].pct_change().dropna().values
    results = []

    for lag in lags:
        shifted = d[["temp_avg_c", "precip_total_mm"]].shift(lag).iloc[1:]
        shifted = shifted.reset_index(drop=True)

        n = min(len(growth), len(shifted))
        if n < 4:
            continue

        pg = detrend(growth[:n])

        for var in ["temp_avg_c", "precip_total_mm"]:
            x = shifted[var].values[:n]
            if np.isnan(x).any():
                continue

            rho, p = spearmanr(pg, detrend(x))
            results.append({
                "lag_years": lag,
                "climate_var": var,
                "spearman_rho": rho,
                "p_value": p,
                "n": n
            })

    return pd.DataFrame(results)

decoupling_table = detrended_lagged_correlation(demo_annual, climate_annual)
decoupling_table.to_csv(
    OUT_DIR / "tables" / "climate_population_correlation.csv", index=False
)

decoupling_table

## 5. Rolling-origin validation

In [ ]:
from scipy.interpolate import CubicSpline

MIN_TRAIN_YEARS = 5

def rolling_origin_splits(years, min_train=MIN_TRAIN_YEARS):
    years = sorted(years)
    for k in range(min_train, len(years)):
        yield years[:k], years[k]

def annual_to_monthly_index(years):
    return np.array([
        y + (m - 0.5) / 12
        for y in years
        for m in range(1, 13)
    ])

def monthly_upsample_train_only(train_years, train_values):
    x = annual_to_monthly_index(train_years)
    cs = CubicSpline(train_years, train_values, extrapolate=True)
    return x, cs(x)

splits = list(rolling_origin_splits(YEARS))
print(f"Independent one-step-ahead validation folds: {len(splits)}")
for train, test in splits:
    print(f"{train[0]}–{train[-1]} -> {test}")

## 6. Forecasting models

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

def fit_polynomial(train_years, train_pop, degree=2):
    pf = PolynomialFeatures(degree=degree, include_bias=False)
    X = pf.fit_transform(np.asarray(train_years).reshape(-1, 1))
    model = LinearRegression().fit(X, train_pop)
    return model, pf

def polynomial_predict(model, pf, year):
    X = pf.transform(np.array([[year]]))
    return float(model.predict(X)[0])

def fit_mlp(train_years, train_pop, hidden=(8, 8), alpha=0.1):
    x_month, y_month = monthly_upsample_train_only(train_years, train_pop)

    xs = StandardScaler()
    ys = StandardScaler()

    X = xs.fit_transform(x_month.reshape(-1, 1))
    Y = ys.fit_transform(y_month.reshape(-1, 1)).ravel()

    model = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=alpha,
        learning_rate_init=1e-3,
        max_iter=2000,
        early_stopping=False,
        random_state=SEED
    )
    model.fit(X, Y)
    return model, xs, ys

def mlp_predict_year(model, xs, ys, year):
    X = xs.transform(np.array([[year + 0.5]]))
    pred = ys.inverse_transform(model.predict(X).reshape(-1, 1))[0, 0]
    return float(max(pred, 0.0))

def fit_best_arimax(train_years, train_pop, max_p=1, max_d=1, max_q=1):
    # Fit on log population to enforce positive forecasts.
    y = pd.Series(
        np.log(np.asarray(train_pop, dtype=float)),
        index=pd.Index(train_years, name="year")
    )

    rows = []
    best = None

    for p in range(max_p + 1):
        for d in range(max_d + 1):
            for q in range(max_q + 1):
                if p == 0 and d == 0 and q == 0:
                    continue
                try:
                    model = SARIMAX(
                        y, order=(p, d, q),
                        seasonal_order=(0, 0, 0, 0),
                        enforce_stationarity=True,
                        enforce_invertibility=True
                    )
                    res = model.fit(disp=False)
                    rows.append({"p": p, "d": d, "q": q, "aic": res.aic, "bic": res.bic})
                    if best is None or res.aic < best[1].aic:
                        best = ((p, d, q), res)
                except Exception:
                    pass

    if best is None:
        raise RuntimeError("No ARIMA specification could be fitted.")

    return best, pd.DataFrame(rows).sort_values("aic").reset_index(drop=True)

def arimax_predict_one(res, year):
    pred_log = float(res.get_forecast(steps=1).predicted_mean.iloc[0])
    return float(np.exp(pred_log))

## 7. Genuine one-step-ahead model comparison

In [ ]:
def run_rolling_origin(demo_df):
    rows = []

    for train_years, test_year in rolling_origin_splits(demo_df.year.tolist()):
        train = demo_df[demo_df.year.isin(train_years)]
        test = demo_df[demo_df.year == test_year]
        y_train = train["population"].values
        actual = float(test["population"].iloc[0])

        # Polynomial
        poly, pf = fit_polynomial(train_years, y_train)
        pred_poly = polynomial_predict(poly, pf, test_year)

        # MLP
        mlp, xs, ys = fit_mlp(train_years, y_train)
        pred_mlp = mlp_predict_year(mlp, xs, ys, test_year)

        # ARIMAX/ARIMA without exogenous predictors
        try:
            (order, ar_res), _ = fit_best_arimax(train_years, y_train)
            pred_arimax = arimax_predict_one(ar_res, test_year)
        except Exception:
            order = None
            pred_arimax = np.nan

        rows.append({
            "test_year": test_year,
            "n_train": len(train_years),
            "actual": actual,
            "pred_polynomial": pred_poly,
            "pred_mlp": pred_mlp,
            "pred_arimax": pred_arimax,
            "arimax_order": order
        })

    return pd.DataFrame(rows)

rolling_results = run_rolling_origin(demo_annual)
rolling_results.to_csv(
    OUT_DIR / "tables" / "rolling_origin_forecast_errors.csv", index=False
)

rolling_results

In [ ]:
def error_summary(df):
    rows = []

    for col in ["pred_polynomial", "pred_mlp", "pred_arimax"]:
        g = df.dropna(subset=[col]).copy()
        if len(g) == 0:
            continue

        err = g["actual"] - g[col]
        rows.append({
            "model": col.replace("pred_", ""),
            "n_forecasts": len(g),
            "MAPE_%": (err.abs() / g["actual"]).mean() * 100,
            "RMSE": np.sqrt(np.mean(err**2)),
            "MAE": err.abs().mean()
        })

    return pd.DataFrame(rows)

summary = error_summary(rolling_results)
summary.to_csv(OUT_DIR / "tables" / "rolling_origin_error_summary.csv", index=False)
summary

## 8. Diebold–Mariano comparison

In [ ]:
from scipy import stats

def diebold_mariano(e1, e2):
    e1 = np.asarray(e1)
    e2 = np.asarray(e2)

    mask = np.isfinite(e1) & np.isfinite(e2)
    d = np.abs(e1[mask])**2 - np.abs(e2[mask])**2
    n = len(d)

    if n < 3 or np.var(d) == 0:
        return np.nan, np.nan, n

    dm = d.mean() / np.sqrt(np.var(d, ddof=0) / n)

    # Harvey–Leybourne–Newbold small-sample correction for h=1
    hln = np.sqrt((n + 1) / n)
    dm_hln = dm * hln
    p = 2 * (1 - stats.t.cdf(abs(dm_hln), df=n - 1))

    return dm_hln, p, n

dm_rows = []
models = ["polynomial", "mlp", "arimax"]

for i, m1 in enumerate(models):
    for m2 in models[i+1:]:
        e1 = rolling_results["actual"] - rolling_results[f"pred_{m1}"]
        e2 = rolling_results["actual"] - rolling_results[f"pred_{m2}"]
        stat, p, n = diebold_mariano(e1, e2)

        dm_rows.append({
            "model_1": m1,
            "model_2": m2,
            "dm_stat": stat,
            "p_value": p,
            "n_independent_forecasts": n
        })

dm_table = pd.DataFrame(dm_rows)
dm_table.to_csv(OUT_DIR / "tables" / "diebold_mariano_results.csv", index=False)
dm_table

## 9. Final 2025–2030 forecasts

In [ ]:
FORECAST_YEARS = list(range(2025, 2031))
all_years = demo_annual.year.tolist()
all_pop = demo_annual.population.values

# Final polynomial model
poly, pf = fit_polynomial(all_years, all_pop)
poly_forecast = {
    y: polynomial_predict(poly, pf, y)
    for y in FORECAST_YEARS
}

# Final MLP model
mlp, xs, ys = fit_mlp(all_years, all_pop)
mlp_forecast = {
    y: mlp_predict_year(mlp, xs, ys, y)
    for y in FORECAST_YEARS
}

# Final ARIMA model on log population
(best_order, arimax_res), arimax_order_table = fit_best_arimax(all_years, all_pop)
arimax_fc = arimax_res.get_forecast(steps=len(FORECAST_YEARS))
arimax_forecast = np.exp(arimax_fc.predicted_mean.values)

final_forecast = pd.DataFrame({
    "year": FORECAST_YEARS,
    "polynomial": [poly_forecast[y] for y in FORECAST_YEARS],
    "mlp": [mlp_forecast[y] for y in FORECAST_YEARS],
    "arimax": arimax_forecast
})

final_forecast.round(0)

In [ ]:
# Bootstrap intervals from genuine one-step-ahead validation errors.
def bootstrap_interval(point, errors, n_boot=5000, seed=SEED):
    errors = np.asarray(errors)
    errors = errors[np.isfinite(errors)]

    if len(errors) < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    sampled = rng.choice(errors, size=n_boot, replace=True)
    forecasts = point + sampled

    return (
        np.percentile(forecasts, 2.5),
        np.percentile(forecasts, 97.5)
    )

for model in ["polynomial", "mlp", "arimax"]:
    errors = rolling_results["actual"] - rolling_results[f"pred_{model}"]
    lo_hi = [bootstrap_interval(v, errors) for v in final_forecast[model]]

    final_forecast[f"{model}_95pi_lo"] = [x[0] for x in lo_hi]
    final_forecast[f"{model}_95pi_hi"] = [x[1] for x in lo_hi]

final_forecast.to_csv(
    OUT_DIR / "tables" / "final_forecast_2025_2030.csv", index=False
)

final_forecast.round(0)

## 10. Final forecast figure

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    demo_annual.year,
    demo_annual.population,
    "ko-",
    linewidth=1.5,
    label="Observed"
)

for model, label in [
    ("polynomial", "Polynomial"),
    ("mlp", "MLP"),
    ("arimax", "ARIMA")
]:
    ax.plot(
        final_forecast.year,
        final_forecast[model],
        "--",
        linewidth=1.5,
        label=label
    )
    ax.fill_between(
        final_forecast.year,
        final_forecast[f"{model}_95pi_lo"],
        final_forecast[f"{model}_95pi_hi"],
        alpha=0.15
    )

ax.set_xlabel("Year")
ax.set_ylabel("Population")
ax.set_title("Soran population: observed and forecast, 2015–2030")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

fig.savefig(
    OUT_DIR / "figures" / "final_forecast.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 11. Final model-selection summary

In [ ]:
print("Observed years:", f"{YEARS[0]}–{YEARS[-1]}")
print("Forecast years:", f"{FORECAST_YEARS[0]}–{FORECAST_YEARS[-1]}")
print("Independent validation forecasts:", len(rolling_results))
print("Selected final ARIMA order:", best_order)
print("\nValidation summary:")
display(summary)

print("\nFinal forecasts:")
display(final_forecast.round(0))

## Output files

The notebook writes the following reproducibility outputs to `D:\Kaka Paper\files\outputs_clean`:

- `tables/climate_annual.csv`
- `tables/population_annual_clean.csv`
- `tables/climate_population_correlation.csv`
- `tables/rolling_origin_forecast_errors.csv`
- `tables/rolling_origin_error_summary.csv`
- `tables/diebold_mariano_results.csv`
- `tables/final_forecast_2025_2030.csv`
- `figures/final_forecast.png`

**Important:** with only 10 annual observations, all model-comparison and forecast results should be interpreted as exploratory/suggestive rather than confirmatory.